# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HaneefAderolu/ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [39]:
# Cell 0 — Setup (run this FIRST every session)
import subprocess
subprocess.run(['pip', 'install', 'duckdb', '-q'], capture_output=True)

import os
import duckdb
import pandas as pd
import numpy as np

# Get HF token from Colab secrets
# (🔑 key icon on left → Add secret → Name: HF_TOKEN → paste your token → toggle ON)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN secret is empty")
    os.environ['HF_TOKEN'] = HF_TOKEN
    print("✓ HF_TOKEN loaded")
except Exception as e:
    print(f"✗ Token error: {e}")
    print("  → Click the 🔑 key icon on the left sidebar, add HF_TOKEN, toggle it ON, re-run")
    raise

# Connect DuckDB
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

try:
    con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
    """)
    print("✓ DuckDB secret created")
except Exception as e:
    if 'already exists' in str(e).lower():
        print("✓ DuckDB secret already exists (safe to ignore on re-run)")
    else:
        raise

# Warehouse paths
BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{BASE}/fact_content_daily_performance"
DIM_C  = f"{BASE}/dim_content/dim_content.parquet"
DIM_CL = f"{BASE}/dim_clients/dim_clients.parquet"

# Quick connection test
try:
    test = con.execute(f"""
        SELECT COUNT(*) AS n
        FROM read_parquet('{FACT}/month=2026-03/*.parquet')
        LIMIT 1
    """).df()
    print(f"✓ Warehouse connected — test row count: {test['n'][0]:,}")
except Exception as e:
    print(f"✗ Warehouse error: {e}")
    print("  → Check that you requested access at huggingface.co/datasets/FlyRank/internship-warehouse")
    print("  → Use a plain READ token, not a fine-grained token")
    raise

print("\nAll ready. con, FACT, DIM_C, DIM_CL are defined. Run the cells below.")

✓ HF_TOKEN loaded
✓ DuckDB secret created
✓ Warehouse connected — test row count: 9,841,378

All ready. con, FACT, DIM_C, DIM_CL are defined. Run the cells below.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
1. What one row means:
   One row = one content page on one report date, for one client.
   The grain is (report_date × client_hash_id × content_hash_id).
   For my analysis I aggregate daily rows up to one row per page per month.

2. Which table(s):
   fact_content_daily_performance, partition month=2026-03.
   I also join dim_content for static page attributes (word count, content type).

3. Time window:
   March 2026 (month=2026-03) is my working month.
   June 2026 (the _sample table) is sealed - I never touch it for feature or label development.

4. What I predict or rank:
   Lane 1 does not predict - it ranks signals.
   The outcome variable is total monthly gsc_impressions per page.
   I rank other signals (position, engagement rate, bounce rate, days observed)
   by their observed correlation with that outcome.

5. One thing deliberately excluded:
   gsc_clicks is excluded from the signal list.
   clicks / impressions = CTR, and CTR is directly downstream of impressions.
   Using clicks as a signal to explain impressions is circular - it would inflate
   every correlation and make the findings meaningless.

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
Five features for Lane 1 - Ranking Signal Analysis

Feature 1: avg_position
  Built as: AVG(gsc_avg_position) over the month, excluding zeros (zero means no data)
  Knowable at decision moment because: GSC reports position for past days only - always historical

Feature 2: avg_engagement_rate
  Built as: AVG(ga4_engagement_rate) where ga4_data_available IS TRUE
  Knowable at decision moment because: GA4 engagement measures sessions that already happened

Feature 3: avg_bounce_rate
  Built as: AVG(ga4_bounce_rate) where ga4_data_available IS TRUE
  Knowable at decision moment because: past visitor behaviour — the session is already over

Feature 4: days_observed
  Built as: COUNT(DISTINCT report_date) for that page in the month
  Knowable at decision moment because: counts how many days the page appeared in GSC reports -
  a proxy for how established the page is, based entirely on past data

Feature 5: total_impressions (the outcome variable - used only as the thing we explain)
  Built as: SUM(gsc_impressions) over the month
  Knowable at decision moment because: sum of past daily impression counts from GSC
  NOTE: this is the outcome we are studying, not a feature we feed into a model

In [41]:
# Build the five-feature frame using real warehouse column names
feature_frame = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(DISTINCT report_date)                             AS days_observed,
    SUM(gsc_impressions)                                    AS total_impressions,
    ROUND(AVG(CASE WHEN gsc_avg_position > 0
              THEN gsc_avg_position END), 2)                AS avg_position,
    ROUND(
        100.0 * SUM(ga4_engaged_sessions)
        / NULLIF(SUM(ga4_users), 0), 3
    )                                                       AS avg_engagement_rate,
    SUM(ga4_pageviews)                                      AS total_pageviews
FROM read_parquet('{FACT}/month=2026-03/*.parquet')
WHERE ga4_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

print(f"Feature frame: {feature_frame.shape[0]:,} rows x {feature_frame.shape[1]} columns")
print(f"One row = one content page in March 2026\n")
print(feature_frame.head(8).to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 90,489 rows x 7 columns
One row = one content page in March 2026

            client_hash_id           content_hash_id  days_observed  total_impressions  avg_position  avg_engagement_rate  total_pageviews
0  client_9958f0a7ae1df715  content_810cf06597918291             19              257.0         11.19               15.000             44.0
1  client_9958f0a7ae1df715  content_b813c73d7000b3b1              7              180.0          8.67               28.571              7.0
2  client_9958f0a7ae1df715  content_5a77dbf5671c5a65             31            19657.0          4.53                9.756            174.0
3  client_9958f0a7ae1df715  content_f5e11209b398d173             10               47.0         10.05               18.182             12.0
4  client_9958f0a7ae1df715  content_8f3fa2db89105948              2                1.0          7.00                0.000              2.0
5  client_9958f0a7ae1df715  content_278030b007943b07             10              319.

In [42]:
# THE TRAP — add a leaky column, watch score jump, then remove it
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

ff = feature_frame.dropna()
y  = ff['total_impressions']

# --- WITH leaky column (gsc_clicks is leaky — clicks/impressions = CTR,
#     which is directly derived from the outcome we are studying) ---
clicks_leaky = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS total_clicks
FROM read_parquet('{FACT}/month=2026-03/*.parquet')
WHERE ga4_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

ff_leaky = ff.merge(clicks_leaky, on=['client_hash_id','content_hash_id'], how='left').dropna()
y_leaky  = ff_leaky['total_impressions']

X_leak = ff_leaky[['avg_position','avg_engagement_rate',
                    'days_observed','total_pageviews','total_clicks']]
Xtr, Xte, ytr, yte = train_test_split(X_leak, y_leaky, test_size=0.2, random_state=42)
m_leak = GradientBoostingRegressor(n_estimators=50, random_state=42).fit(Xtr, ytr)
r2_leak = r2_score(yte, m_leak.predict(Xte))
print(f"R² WITH leaky column  (total_clicks): {r2_leak:.3f}  ← looks great, means nothing")

# --- WITHOUT leaky column (honest) ---
X_honest = ff[['avg_position','avg_engagement_rate','days_observed','total_pageviews']]
Xtr2, Xte2, ytr2, yte2 = train_test_split(X_honest, y, test_size=0.2, random_state=42)
m2 = GradientBoostingRegressor(n_estimators=50, random_state=42).fit(Xtr2, ytr2)
r2_honest = r2_score(yte2, m2.predict(Xte2))
print(f"R² WITHOUT leaky column (honest)    : {r2_honest:.3f}  ← this is what we keep")

print(f"\nLeaky column removed. Inflation was {r2_leak - r2_honest:.3f} points.")
print("The honest number is what gets reported.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

R² WITH leaky column  (total_clicks): 0.539  ← looks great, means nothing
R² WITHOUT leaky column (honest)    : 0.480  ← this is what we keep

Leaky column removed. Inflation was 0.059 points.
The honest number is what gets reported.


In [43]:
# Query 1 — Grain: should return 0 rows if grain holds
grain_check = con.execute(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM read_parquet('{FACT}/month=2026-03/*.parquet')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING c > 1
LIMIT 5
""").df()

print(f"Duplicate grain rows (expect 0): {len(grain_check)}")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows (expect 0): 0
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [44]:
# Query 1 — Grain: should return 0 rows if grain holds
grain_check = con.execute(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM read_parquet('{FACT}/month=2026-03/*.parquet')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING c > 1
LIMIT 5
""").df()

print(f"Duplicate grain rows (expect 0): {len(grain_check)}")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows (expect 0): 0
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []


In [45]:
# Query 2 - How many rows, which dates, how many pages and clients
fact1 = con.execute(f"""
SELECT
    COUNT(*)                            AS total_rows,
    COUNT(DISTINCT content_hash_id)     AS unique_pages,
    COUNT(DISTINCT client_hash_id)      AS unique_clients,
    MIN(report_date)                    AS earliest_date,
    MAX(report_date)                    AS latest_date
FROM read_parquet('{FACT}/month=2026-03/*.parquet')
""").df()

print("Row count and date span for month=2026-03:")
print(fact1.to_string())

Row count and date span for month=2026-03:
   total_rows  unique_pages  unique_clients earliest_date latest_date
0     9841378        331437              55    2026-03-01  2026-03-31


In [46]:
# Query 3 — How many rows survive the ga4_data_available IS TRUE filter
fact2 = con.execute(f"""
SELECT
    COUNT(*)                                                                AS total_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)            AS ga4_available_rows,
    ROUND(
        100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
        / COUNT(*), 1
    )                                                                       AS pct_available
FROM read_parquet('{FACT}/month=2026-03/*.parquet')
""").df()

print("GA4 availability - rows surviving ga4_data_available IS TRUE:")
print(fact2.to_string())

GA4 availability — rows surviving ga4_data_available IS TRUE:
   total_rows  ga4_available_rows  pct_available
0     9841378            413966.0            4.2


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
Named limitation: uneven history depth across clients.

Some clients have GSC data going back to January 2025 (17 months of history).
Others started much later and have only a few weeks.

This means days_observed for a new client's page maxes out at ~10
while an established client's page can have 30+ days in the same month.
Any signal involving time (days observed, trend slope, age) behaves
differently per client and must be interpreted per client, not globally.

A single global threshold applied across all clients will silently mix
long-established pages with brand-new ones and produce misleading findings.

In [47]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.